# CellRank

In [ ]:
import os
os.chdir(path_to_wd)

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import sys
import scipy
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.sparse import csr_matrix, csc_matrix, coo_matrix, lil_matrix
from scipy import io
import cellrank as cr
import palantir
from cellrank.kernels import PseudotimeKernel, ConnectivityKernel
from cellrank.estimators import GPCCA
from cellrank import Lineage
from muon import prot as pt
from pandas.api.types import CategoricalDtype
import matplotlib.pyplot as plt
import pickle
import mudata
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import ListedColormap

plt.rcParams['font.family']=['Dejavu serif']
plt.rcParams['figure.dpi'] = 100
plt.rcParams['pdf.fonttype'] = 'truetype'
sc.settings.set_figure_params(frameon=False, dpi=100)
cr.settings.verbosity = 2

In [ ]:
adata_full = sc.read("Reproducibility/Data/DOGMA/UC_DOGMA_RNA_ATAC_ADT_after_QC.h5ad")
adata_RNA = adata_full[:, adata_full.var.modality == "Gene Expression"].copy()

In [ ]:
# color map
blue_yellow_colors = ["#352A86","#343DAE","#0262E0","#1389D2","#2DB7A3","#A5BE6A","#F8BA43","#F6DA23","#F8FA0D"]
horizon_colors = ["#000033","#000075","#0000B6","#0000F8","#2E00FF","#6100FF","#9408F7","#C729D6","#FA4AB5",
                 "#FF6A95","#FF8B74","#FFAC53","#FFCD32","#FFEE11","#FFFF60"]
solar_extra_colors = ["#3361A5","#248AF3","#14B3FF","#88CEEF","#C1D5DC","#EAD397","#FDB31A","#E42A2A","#A31D1D"]

blue_yellow = LinearSegmentedColormap.from_list('custom',blue_yellow_colors)
horizon = LinearSegmentedColormap.from_list('custom',horizon_colors)
solar_extra = LinearSegmentedColormap.from_list('custom',solar_extra_colors)

## B

In [ ]:
tmp_subset = 'B'

fig_dir = f"Reproducibility/Results/Plots/{tmp_subset}/"
os.makedirs(fig_dir, exist_ok = True)

adata = adata_RNA[adata_RNA.obs['lineage']==tmp_subset].copy()

tmp_path = f"Reproducibility/Data/embeddings/UC_DOGMA_{tmp_subset}_latent_df.txt"
latent_df = pd.read_csv(tmp_path,  sep='\t', index_col = 0)
latent_df = latent_df.loc[adata.obs_names, : ].values 
adata.obsm["MultiVI_latent"] = latent_df

tmp_path = f"Reproducibility/Data/embeddings/UC_DOGMA_{tmp_subset}_UMAP_df.txt"
UMAP_df = pd.read_csv(tmp_path,  sep='\t', index_col = 0)
adata.obsm["X_umap"] = UMAP_df.values

sc.pp.neighbors(adata, use_rep="MultiVI_latent")

In [ ]:
cat_type = CategoricalDtype(categories=["B_naive","B_memory",'Atypical_B',"GC_B","Plasma"], ordered=True)
adata.obs['celltype'] = adata.obs['celltype'].astype(cat_type)

### Diffusion pseudotime

In [ ]:
adata = adata[adata.obs['celltype'].isin(["B_memory","Atypical_B"])].copy()

sc.pp.normalize_total(adata, target_sum=1e4)
palantir.preprocess.log_transform(adata)

adata.layers['scaled'] = sc.pp.scale(adata, copy=True).X

In [ ]:
# Run diffusion maps
sc.tl.diffmap(adata)
sc.pl.scatter(
    adata,
    basis="diffmap",
    color=["celltype"],
    components=[0, 1],
    frameon=True,
    show=True
)

In [ ]:
# Pseudotime
adata_tmp = adata[adata.obs["celltype"].isin(["B_memory"])].copy()
root_ixs = adata_tmp.obsm["X_diffmap"][:, 1].argmin()
root_cell = adata_tmp.obs_names[root_ixs]
adata.uns['iroot'] = np.flatnonzero(adata.obs_names == root_cell)[0]
sc.tl.dpt(adata)

In [ ]:
# Build CellRank kernels
# Pseudotime-based directional kernel
pt_kernel = PseudotimeKernel(adata, time_key='dpt_pseudotime').compute_transition_matrix()

# Similarity-based kernel from MultiVI latent
conn_kernel = ConnectivityKernel(adata).compute_transition_matrix()

# Combine kernels
tmp_ratio = 0.2
combined_kernel = tmp_ratio * conn_kernel + (1-tmp_ratio) * pt_kernel

combined_kernel.write_to_adata()

In [ ]:
sc.set_figure_params(figsize=(4, 4)) 
sc.set_figure_params(vector_friendly=True, dpi_save=100)
sc.pl.embedding(
    adata,
    basis="umap",
    color=['dpt_pseudotime'],
    color_map="gnuplot",
    legend_loc="on data",
    show = False
)
plt.savefig(f"{fig_dir}FigureS10D_UMAP_pseudotime.pdf", bbox_inches='tight')
plt.close()

In [ ]:
adata.obs.to_csv("Reproducibility/Results/CellRank2/B/Atypical_B_dpt_pseudotime.txt", sep='\t')

In [ ]:
g = cr.estimators.GPCCA(combined_kernel)
g.compute_schur()
g.compute_macrostates(n_states=2, cluster_key="celltype")

In [ ]:
g.set_terminal_states(states=["Atypical_B"])
g.set_initial_states(states=["B_memory"])  
g.compute_fate_probabilities()

### RNA

In [ ]:
model = cr.models.GAMR(adata, smoothing_penalty=10.0)

In [ ]:
cr.pl.gene_trends(
    adata,
    model=model,
    data_key='scaled',
    genes=['CD86', 'ICAM1', 'FAS', 'LGALS9'],
    same_plot=True,
    ncols=1,
    sharex = 'row',
    sharey = 'col',
    time_key="dpt_pseudotime",
    hide_cells=True,
)
plt.savefig(f"{fig_dir}FigureS10D_RNA_marker_activation.pdf", bbox_inches='tight')
plt.close()

In [ ]:
cr.pl.gene_trends(
    adata,
    model=model,
    data_key='scaled',
    genes=['TBX21','ZEB2','ITGAX','FCRL4','FCRL5'],
    same_plot=True,
    ncols=1,
    sharex = 'row',
    sharey = 'col',
    time_key="dpt_pseudotime",
    hide_cells=True,
)
plt.savefig(f"{fig_dir}FigureS10D_RNA_marker_atypical_B.pdf", bbox_inches='tight')
plt.close()

In [ ]:
cr.pl.gene_trends(
    adata,
    model=model,
    data_key='scaled',
    genes=['CCR1', 'CXCR3', 'CXCR5', 'IFNGR1', 'IL21R'],
    same_plot=True,
    ncols=1,
    sharex = 'row',
    sharey = 'col',
    time_key="dpt_pseudotime",
    hide_cells=True,
)
plt.savefig(f"{fig_dir}FigureS10D_RNA_marker_cytokine_receptor.pdf", bbox_inches='tight')
plt.close()

### Protein

In [ ]:
adt_df = adata.obsm["protein_expression"].copy()
adt_df.columns = [col.split("-")[-1] for col in adt_df.columns]

pro_adata = sc.AnnData(adt_df.copy(), obs=adata.obs)
pro_adata.layers["counts"] = pro_adata.X.copy()
pro_adata.obsm["X_umap"] = adata.obsm["X_umap"]
pro_adata.obsm["MultiVI_latent"] = adata.obsm["MultiVI_latent"]

pro_adata.X = pro_adata.X.astype(float)
pt.pp.clr(pro_adata)

pro_adata.layers['scaled'] = sc.pp.scale(pro_adata, copy=True).X
pro_adata.obsm['lineages_fwd'] = adata.obsm['lineages_fwd']

In [ ]:
model_pro = cr.models.GAMR(pro_adata, smoothing_penalty=10.0)

In [ ]:
selected_adts = ['CD86', 'CD11c', 'HLADR', 'FcRL4']

cr.pl.gene_trends(
    pro_adata,
    model=model_pro,
    data_key="scaled",
    genes=selected_adts,
    same_plot=True,
    sharex = 'row',
    sharey = 'col',
    ncols=1,
    time_key="dpt_pseudotime",
    hide_cells=True
)
plt.savefig(f"{fig_dir}FigureS10D_Protein_marker_atypical_B.pdf", bbox_inches='tight')
plt.close()

### eRegulon

In [ ]:
scplus_mdata = mudata.read(f"Reproducibility/Results/scenicplus/{tmp_subset}/scplusmdata.h5mu")
eRegulon_gene_AUC = ad.concat(
    [scplus_mdata["direct_gene_based_AUC"], scplus_mdata["extended_gene_based_AUC"]],
    axis = 1,
)
eRegulon_gene_AUC.obs = scplus_mdata.obs.loc[eRegulon_gene_AUC.obs_names]

In [ ]:
eRegulon_gene_AUC.obs_names = eRegulon_gene_AUC.obs_names.str.replace('___cisTopic', '', regex=False)
eRegulon_tmp = eRegulon_gene_AUC[eRegulon_gene_AUC.obs['scRNA_counts:celltype'].isin(['B_memory', 'Atypical_B'])].copy()

In [ ]:
# metacell selection
metadata = pd.read_csv(f"Reproducibility/Results/SEACells/{tmp_subset}/UC_DOGMA_{tmp_subset}_metadata_w_SEACells.txt", sep='\t', index_col=0)
cell_use = np.intersect1d(adata.obs_names, metadata.index)
metadata = metadata.loc[cell_use,:]

adata_tmp = adata[cell_use,:].copy()
adata_tmp.obs['SEACell'] = metadata.loc[cell_use,'SEACell']

metacell_use = np.intersect1d(adata_tmp.obs['SEACell'], eRegulon_tmp.obs_names)
adata_tmp = adata_tmp[adata_tmp.obs['SEACell'].isin(metacell_use)].copy()

In [ ]:
# Aggregate pseudotime values
avg_pseudotime = adata_tmp.obs['dpt_pseudotime'].groupby(adata_tmp.obs['SEACell']).mean().loc[metacell_use].values
eRegulon_tmp = eRegulon_tmp[metacell_use,:]
eRegulon_tmp.obs['dpt_pseudotime'] = avg_pseudotime

In [ ]:
# Convert the original Lineage object to DataFrame
lineage_df = pd.DataFrame(
    adata_tmp.obsm['lineages_fwd'],
    index=adata_tmp.obs_names,
    columns=['Atypical_B']
)

# Aggregate by metacells
avg_lineage_df = (
    lineage_df
    .groupby(adata_tmp.obs['SEACell'])
    .mean()
    .loc[metacell_use]  # use only the SEACells present in rna_ad
)

# Convert back to Lineage object
avg_lineages = Lineage(avg_lineage_df.values, names=avg_lineage_df.columns.tolist())

eRegulon_tmp.obsm['lineages_fwd'] = avg_lineages
eRegulon_tmp.layers['scaled'] = sc.pp.scale(eRegulon_tmp, copy=True).X

In [ ]:
model_meta = cr.models.GAMR(eRegulon_tmp, n_knots=4, smoothing_penalty=10.0)

In [ ]:
cr.pl.gene_trends(
    eRegulon_tmp,
    model=model_meta,
    data_key='scaled',
    genes=['ELF2_extended_+/+_(406g)','KLF2_extended_+/+_(21g)'],
    same_plot=True,
    ncols=5,
    time_key="dpt_pseudotime",
    hide_cells=True,
)
plt.savefig(f"{fig_dir}FigureS10D_eRegulon_naive_quiescence.pdf", bbox_inches='tight')
plt.close()

## CD4<sup>+</sup> T

In [ ]:
tmp_subset = 'CD4_T'

fig_dir = f"Reproducibility/Results/Plots/{tmp_subset}/"
os.makedirs(fig_dir, exist_ok = True)

adata = sc.read("Reproducibility/Data/DOGMA/UC_DOGMA_RNA_ADT_Treg_wo_cycling.h5ad")

In [ ]:
cat_type = CategoricalDtype(categories=["CD4_Tn","CD4_Tcm",'CD4_Tsen',"CD4_T_CD26","CD4_CTL","CD4_Th17","CD4_Tfh-like",
                                        "CD4_Tph-like","CD4_T_proliferative","Treg_naive","Treg_effector"], ordered=True)
adata.obs['celltype'] = adata.obs['celltype'].astype(cat_type)

### Diffusion pseudotime

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
palantir.preprocess.log_transform(adata)

dm_input = pd.DataFrame(adata.obsm['MultiVI_latent'], index=adata.obs_names)
dm_res = palantir.utils.run_diffusion_maps(dm_input, n_components=5)
imputed_X = palantir.utils.run_magic_imputation(adata, dm_res=dm_res)

In [ ]:
# Run diffusion maps
sc.tl.diffmap(adata)

sc.pl.scatter(
    adata,
    basis="diffmap",
    color=["celltype"],
    components=[0, 1],
    frameon=True,
    show=True
)

# root identification
root_ixs = adata.obsm["X_diffmap"][:, 1].argmax()
root_cell = adata.obs_names[root_ixs]

initial_states = pd.Series(
    ["Treg_naive"],
    index=[root_cell],
)

palantir.plot.highlight_cells_on_umap(adata, initial_states)
plt.show()

# Pseudotime
adata.uns['iroot'] = np.flatnonzero(adata.obs_names == root_cell)[0]
sc.tl.dpt(adata)

In [ ]:
# Build CellRank kernels
# Pseudotime-based directional kernel
pt_kernel = PseudotimeKernel(adata, time_key='dpt_pseudotime').compute_transition_matrix()

# Similarity-based kernel from MultiVI latent
conn_kernel = ConnectivityKernel(adata).compute_transition_matrix()

# Combine kernels
tmp_ratio = 0.2
combined_kernel = tmp_ratio * conn_kernel + (1-tmp_ratio) * pt_kernel

combined_kernel.write_to_adata()

In [ ]:
adata.obs.to_csv("Reproducibility/Results/CellRank2/CD4_T/Treg_dpt_pseudotime.txt", sep='\t')

In [ ]:
g = cr.estimators.GPCCA(combined_kernel)
g.compute_schur()
g.compute_macrostates(n_states=4, cluster_key="celltype")
g.plot_macrostates(which="all", legend_loc="right", s=100)

In [ ]:
# Semi-automatically terminal setting
# 'Treg_effector_2' = Terminal state 1, 'Treg_effector_3' = Terminal state 2
g.set_initial_states(states=['Treg_naive']) 
g.set_terminal_states(states=['Treg_effector_2', 'Treg_effector_3'])

Compute fate probabilities

In [ ]:
g.compute_fate_probabilities()
g.plot_fate_probabilities(same_plot = True)

fate_probs = g.fate_probabilities
fate_probs_df = pd.DataFrame(fate_probs.X, index=adata.obs_names, columns=fate_probs.names)
obs_out = pd.DataFrame(adata.obsm['X_umap'], index=adata.obs_names, columns=['UMAP_1', 'UMAP_2'])
obs_out = obs_out.join(fate_probs_df)
obs_out.to_csv("Reproducibility/Results/CellRank2/CD4_T/fate_probabilities_dpt_Treg.csv")

### RNA

In [ ]:
model = cr.models.GAMR(adata)

In [ ]:
# Fig.5D
cr.pl.gene_trends(
    adata,
    model=model,
    data_key='MAGIC_imputed_data',
    genes=["TBX21", "CXCR3", 'TNFRSF9','IL2RA'],
    same_plot=True,
    time_range=[(0.5, 1.0), (0.5, 1.0)], 
    ncols=2,
    lineage_cmap = ["#fdae61", "#a6d96a"],
    time_key="dpt_pseudotime",
    hide_cells=True,
)

plt.savefig(f"{fig_dir}Figure5D_RNA.pdf", bbox_inches='tight')
plt.close()

In [ ]:
# Fig.S11G
cr.pl.gene_trends(
    adata,
    model=model,
    data_key='MAGIC_imputed_data',
    genes=["HAVCR2","IFI44L",'HSPA1A','TNFRSF18','TNFRSF4','FOXP3'],
    same_plot=True,
    time_range=[(0.5, 1.0), (0.5, 1.0)], 
    ncols=3,
    lineage_cmap = ["#fdae61", "#a6d96a"],
    time_key="dpt_pseudotime",
    hide_cells=True,
)

plt.savefig(f"{fig_dir}FigureS11G_RNA.pdf", bbox_inches='tight')
plt.close()

### Protein

In [ ]:
adt_df = adata.obsm["protein_expression"].copy()
adt_df.columns = [col.split("-")[-1] for col in adt_df.columns]

pro_adata = sc.AnnData(adt_df.copy(), obs=adata.obs)
pro_adata.layers["counts"] = pro_adata.X.copy()
pro_adata.obsm["X_umap"] = adata.obsm["X_umap"]
pro_adata.obsm["MultiVI_latent"] = adata.obsm["MultiVI_latent"]

pro_adata.X = pro_adata.X.astype(float)
pt.pp.clr(pro_adata)

pro_adata.layers['scaled'] = sc.pp.scale(pro_adata, copy=True).X
pro_adata.obsm['lineages_fwd'] = adata.obsm['lineages_fwd']

In [ ]:
model_pro = cr.models.GAMR(pro_adata, n_knots=3, smoothing_penalty=10.0)

In [ ]:
# Fig.5D
selected_adts = ['PD1','CD73',"41BB","CD25"]

cr.pl.gene_trends(
    pro_adata,
    model=model_pro,
    data_key="X",
    genes=selected_adts,
    same_plot=True,
    time_range=[(0.5, 1.0), (0.5, 1.0)], 
    ncols=2,
    lineage_cmap = ["#fdae61", "#a6d96a"],
    time_key="dpt_pseudotime",
    hide_cells=True,
)

plt.savefig(f"{fig_dir}Figure5D_Protein.pdf", bbox_inches='tight')
plt.close()

### TF activity

In [ ]:
# TF_activity
tmp_path = "Reproducibility/Results/LINGER/Primary/cell_population_TF_activity_CD4_T.txt"
TF_activity = pd.read_csv(tmp_path, index_col=0, sep="\t")

adata_TF = sc.AnnData(TF_activity.T.loc[adata.obs_names,:].copy(), obs=adata.obs)
sc.pp.scale(adata_TF, max_value=10)
adata_TF.obsm['lineages_fwd'] = adata.obsm['lineages_fwd']

In [ ]:
model_TF = cr.models.GAMR(adata_TF,n_knots=3, smoothing_penalty=10.0)

In [ ]:
# Fig.5D
cr.pl.gene_trends(
    adata_TF,
    model=model_TF,
    data_key="X",
    genes=["TBX21",'BATF'], 
    same_plot=True,
    time_range=[(0.5, 1.0), (0.5, 1.0)], 
    ncols=3,
    lineage_cmap = ["#fdae61", "#a6d96a"],
    time_key="dpt_pseudotime",
    hide_cells=True,
)

plt.savefig(f"{fig_dir}Figure5D_TF_activity.pdf", bbox_inches='tight')
plt.close()

In [ ]:
# Fig.S11G
cr.pl.gene_trends(
    adata_TF,
    model=model_TF,
    data_key="X",
    genes=["CTCF",'REL'], 
    same_plot=True,
    time_range=[(0.5, 1.0), (0.5, 1.0)], 
    ncols=3,
    lineage_cmap = ["#fdae61", "#a6d96a"],
    time_key="dpt_pseudotime",
    hide_cells=True,
)

plt.savefig(f"{fig_dir}FigureS11G_TF_activity.pdf", bbox_inches='tight')
plt.close()

### eRegulon

In [ ]:
scplus_mdata = mudata.read(f"Reproducibility/Results/scenicplus/{tmp_subset}/scplusmdata.h5mu")
eRegulon_gene_AUC = ad.concat(
    [scplus_mdata["direct_gene_based_AUC"], scplus_mdata["extended_gene_based_AUC"]],
    axis = 1,
)
eRegulon_gene_AUC.obs = scplus_mdata.obs.loc[eRegulon_gene_AUC.obs_names]

In [ ]:
eRegulon_gene_AUC.obs_names = eRegulon_gene_AUC.obs_names.str.replace('___cisTopic', '', regex=False)
eRegulon_tmp = eRegulon_gene_AUC[eRegulon_gene_AUC.obs['scRNA_counts:celltype'].isin(['Treg_naive', 'Treg_effector'])].copy()

In [ ]:
# metacell selection
metadata = pd.read_csv(f"Reproducibility/Results/SEACells/{tmp_subset}/UC_DOGMA_{tmp_subset}_metadata_w_SEACells.txt", sep='\t', index_col=0)
cell_use = np.intersect1d(adata.obs_names, metadata.index)
metadata = metadata.loc[cell_use,:]

adata_tmp = adata[cell_use,:].copy()
adata_tmp.obs['SEACell'] = metadata.loc[cell_use,'SEACell']

metacell_use = np.intersect1d(adata_tmp.obs['SEACell'], eRegulon_tmp.obs_names)
adata_tmp = adata_tmp[adata_tmp.obs['SEACell'].isin(metacell_use)].copy()

In [ ]:
# Aggregate pseudotime values
avg_pseudotime = adata_tmp.obs['dpt_pseudotime'].groupby(adata_tmp.obs['SEACell']).mean().loc[metacell_use].values
eRegulon_tmp = eRegulon_tmp[metacell_use,:]
eRegulon_tmp.obs['dpt_pseudotime'] = avg_pseudotime

In [ ]:
# Convert the original Lineage object to DataFrame
lineage_df = pd.DataFrame(
    adata_tmp.obsm['lineages_fwd'],
    index=adata_tmp.obs_names,
    columns=['Treg_effector_2', 'Treg_effector_3']
)

# Aggregate by metacells
avg_lineage_df = (
    lineage_df
    .groupby(adata_tmp.obs['SEACell'])
    .mean()
    .loc[metacell_use]  # use only the SEACells present in rna_ad
)

# Convert back to Lineage object
avg_lineages = Lineage(avg_lineage_df.values, names=avg_lineage_df.columns.tolist())

eRegulon_tmp.obsm['lineages_fwd'] = avg_lineages
eRegulon_tmp.layers['scaled'] = sc.pp.scale(eRegulon_tmp, copy=True).X

In [ ]:
model_meta = cr.models.GAMR(eRegulon_tmp, n_knots=3, smoothing_penalty=10.0)

In [ ]:
# Fig.5D
cr.pl.gene_trends(
    eRegulon_tmp,
    model=model_meta,
    data_key="X",
    genes=['TBX21_extended_+/+_(29g)','ETS1_extended_+/+_(309g)','BATF_extended_+/+_(396g)','REL_extended_+/+_(267g)'], 
    same_plot=True,
    time_range=[(0.5, 1.0), (0.5, 1.0)], 
    ncols=2,
    lineage_cmap = ["#fdae61", "#a6d96a"],
    time_key="dpt_pseudotime",
    hide_cells=True,
)

plt.savefig(f"{fig_dir}Figure5D_eRegulon.pdf", bbox_inches='tight')
plt.close()

In [ ]:
# Fig.S11G
cr.pl.gene_trends(
    eRegulon_tmp,
    model=model_meta,
    data_key="X",
    genes=['CTCF_extended_+/+_(1223g)','YY1_extended_+/+_(120g)'], 
    same_plot=True,
    time_range=[(0.5, 1.0), (0.5, 1.0)], 
    ncols=1,
    lineage_cmap = ["#fdae61", "#a6d96a"],
    time_key="dpt_pseudotime",
    hide_cells=True,
)
plt.savefig(f"{fig_dir}FigureS11G_eRegulon.pdf", bbox_inches='tight')
plt.close()

## CD8<sup>+</sup> T

In [ ]:
tmp_subset = 'CD8_T_NK_ILC'

fig_dir = f"Reproducibility/Results/Plots/{tmp_subset}/"
os.makedirs(fig_dir, exist_ok = True)

adata_tmp = adata_RNA[adata_RNA.obs['lineage']==tmp_subset].copy()

tmp_path = f"Reproducibility/Data/embeddings/UC_DOGMA_{tmp_subset}_latent_df.txt"
latent_df = pd.read_csv(tmp_path,  sep='\t', index_col = 0)
latent_df = latent_df.loc[adata_tmp.obs_names, : ].values 
adata_tmp.obsm["MultiVI_latent"] = latent_df

tmp_path = f"Reproducibility/Data/embeddings/UC_DOGMA_{tmp_subset}_UMAP_df.txt"
UMAP_df = pd.read_csv(tmp_path,  sep='\t', index_col = 0)
adata_tmp.obsm["X_umap"] = UMAP_df.values

In [ ]:
adata_CD8 = adata_tmp[adata_tmp.obs['celltype'].isin(["CD8_Tn","CD8_Tcm","CD8_Tem","CD8_Temra","CD8_Trm",
                                        "CD8_Tex_1","CD8_Tex_2"])].copy()
sc.pp.neighbors(adata_CD8, use_rep='MultiVI_latent', n_neighbors=30)

In [ ]:
adata_NK = adata_tmp[adata_tmp.obs['celltype'].isin(["NK_CD56_CD49a_Hi_CD103_Hi",'NK_CD56_dim',
                                   "NK_CD56_CD49a_Hi_CD103_Lo","NK_CD56_CD49a_Lo"])].copy()
sc.pp.neighbors(adata_NK, use_rep='MultiVI_latent', n_neighbors=30)

### Diffusion pseudotime

In [ ]:
adata = adata_CD8.copy()

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
palantir.preprocess.log_transform(adata)

dm_input = pd.DataFrame(adata.obsm['MultiVI_latent'], index=adata.obs_names)
dm_res = palantir.utils.run_diffusion_maps(dm_input, n_components=5)
imputed_X = palantir.utils.run_magic_imputation(adata, dm_res=dm_res)

In [ ]:
# Run diffusion maps
sc.tl.diffmap(adata)

adata_CD8_Tn = adata[adata.obs["celltype"].isin(['CD8_Tn'])].copy()
root_ixs = adata_CD8_Tn.obsm["X_diffmap"][:, 4].argmin()
root_cell = adata_CD8_Tn.obs_names[root_ixs]

adata.uns['iroot'] = np.flatnonzero(adata.obs_names == root_cell)[0]
sc.tl.dpt(adata)

In [ ]:
# Build CellRank kernels
# Pseudotime-based directional kernel
pt_kernel = PseudotimeKernel(adata, time_key='dpt_pseudotime').compute_transition_matrix()

# Similarity-based kernel from MultiVI latent
conn_kernel = ConnectivityKernel(adata).compute_transition_matrix()

# Combine kernels
tmp_ratio = 0.2
combined_kernel = tmp_ratio * conn_kernel + (1-tmp_ratio) * pt_kernel

combined_kernel.write_to_adata()

In [ ]:
from pandas.api.types import CategoricalDtype

cat_type = CategoricalDtype(categories=["CD8_Tn","CD8_Tcm",'CD8_Tem',"CD8_Temra","CD8_Trm","CD8_Tex_1","CD8_Tex_2"], ordered=True)
adata.obs['celltype'] = adata.obs['celltype'].astype(cat_type)

celltype_colors = [
    '#1aafc9',  # ocean blue
    '#ff6f61',  # coral
    '#6a9f58',  # olive green
    '#f2b134',  # goldenrod
    '#d883b7',  # muted magenta
    '#6b4c9a',  # indigo (keep this one)
    '#c0592c',  # copper
]

# Fig.S10E
sc.set_figure_params(figsize=(4, 4)) 
sc.set_figure_params(vector_friendly=True, dpi_save=100)
sc.pl.umap(
    adata,
    color=['celltype'],
    palette=celltype_colors,
    frameon=False,
    legend_fontsize=7,
    show = False
)
plt.savefig(f"{fig_dir}FigureS10E_celltype.pdf", bbox_inches='tight')
plt.close()

In [ ]:
sc.set_figure_params(figsize=(4, 4)) 
sc.set_figure_params(vector_friendly=True, dpi_save=100)
sc.pl.embedding(
    adata,
    basis="umap",
    color=['dpt_pseudotime'],
    color_map="gnuplot",
    legend_loc="on data",
    show = False
)
plt.savefig(f"{fig_dir}FigureS10E_pseudotime.pdf", bbox_inches='tight')
plt.close()

In [ ]:
g = cr.estimators.GPCCA(combined_kernel)
g.compute_schur()
g.compute_macrostates(n_states=14, cluster_key="celltype")
g.plot_macrostates(which="all", legend_loc="right", s=100)

In [ ]:
# Semi-automatically terminal setting
g.set_initial_states(states=['CD8_Tn']) 
g.set_terminal_states(states=['CD8_Temra_2', 'CD8_Tex_2_2'])

Compute fate probabilities

In [ ]:
g.compute_fate_probabilities()
g.plot_fate_probabilities(same_plot = True)

fate_probs = g.fate_probabilities
fate_probs_df = pd.DataFrame(fate_probs.X, index=adata.obs_names, columns=fate_probs.names)
obs_out = pd.DataFrame(adata.obsm['X_umap'], index=adata.obs_names, columns=['UMAP_1', 'UMAP_2'])
obs_out = obs_out.join(fate_probs_df)
obs_out.to_csv("Reproducibility/Results/CellRank2/CD8_T_NK_ILC/fate_probabilities_dpt_CD8.csv")

### RNA

In [ ]:
model = cr.models.GAMR(adata, n_knots=5, smoothing_penalty=10.0)

In [ ]:
naive_to_exhaustion_genes = [
    "SELL", "CCR7", "TCF7", "LEF1", "IL7R", "CXCR3", "CD28",
    "EOMES", "KLRG1", "TNF", "CX3CR1", "IFNG", "GZMA", 
    "TOX", "PDCD1", "CTLA4", "HAVCR2", "TIGIT", "ENTPD1"
]

cr.pl.heatmap(
    adata,
    model=model,
    data_key="MAGIC_imputed_data",
    genes=naive_to_exhaustion_genes,
    lineages=["CD8_Tex_2_2"],
    time_key="dpt_pseudotime",
    cbar=True,
    show_all_genes=True,
)

plt.savefig(f"{fig_dir}FigureS10F_RNA_exhaustion_pathway.pdf", bbox_inches='tight')
plt.close()

In [ ]:
Temra_development_genes = [
    "SELL", "CCR7", "TCF7", "LEF1", "IL7R", "CD28",
    "EOMES", "TNF", "TBX21", "ZEB2", "IL2RB", "NKG7",
    "KLRG1", "CX3CR1", "GNLY", "IFNG", "GZMA", "PRF1", "ITGB1" 
]

cr.pl.heatmap(
    adata,
    model=model,
    data_key="MAGIC_imputed_data",
    genes=Temra_development_genes,
    lineages=["CD8_Temra_2"],
    time_key="dpt_pseudotime",
    cbar=True,
    show_all_genes=True,
)

plt.savefig(f"{fig_dir}FigureS10F_RNA_Temra_pathway.pdf", bbox_inches='tight')
plt.close()

### Protein

In [ ]:
adt_df = adata.obsm["protein_expression"].copy()
adt_df.columns = [col.split("-")[-1] for col in adt_df.columns]

pro_adata = sc.AnnData(adt_df.copy(), obs=adata.obs)
pro_adata.layers["counts"] = pro_adata.X.copy()
pro_adata.obsm["X_umap"] = adata.obsm["X_umap"]
pro_adata.obsm["MultiVI_latent"] = adata.obsm["MultiVI_latent"]

pro_adata.X = pro_adata.X.astype(float)
pt.pp.clr(pro_adata)

pro_adata.layers['scaled'] = sc.pp.scale(pro_adata, copy=True).X
pro_adata.obsm['lineages_fwd'] = adata.obsm['lineages_fwd']

In [ ]:
model_pro = cr.models.GAMR(pro_adata, n_knots=4, smoothing_penalty=10.0)

In [ ]:
# Fig.5J
selected_adts = ["ITGAE", "ITGB7"]

cr.pl.gene_trends(
    pro_adata,
    model=model_pro,
    data_key="X",
    genes=selected_adts,
    same_plot=True,
    ncols=2,
    lineage_cmap = ["#4d9221","#c51b7d"],
    time_key="dpt_pseudotime",
    hide_cells=True,
)

plt.savefig(f"{fig_dir}Figure5J_Protein.pdf", bbox_inches='tight')
plt.close()

### eRegulon

In [ ]:
scplus_mdata = mudata.read(f"Reproducibility/Results/scenicplus/{tmp_subset}/scplusmdata.h5mu")
eRegulon_gene_AUC = ad.concat(
    [scplus_mdata["direct_gene_based_AUC"], scplus_mdata["extended_gene_based_AUC"]],
    axis = 1,
)
eRegulon_gene_AUC.obs = scplus_mdata.obs.loc[eRegulon_gene_AUC.obs_names]

In [ ]:
eRegulon_gene_AUC.obs_names = eRegulon_gene_AUC.obs_names.str.replace('___cisTopic', '', regex=False)
eRegulon_tmp = eRegulon_gene_AUC[eRegulon_gene_AUC.obs['scRNA_counts:celltype'].isin(["CD8_Tn","CD8_Tcm",'CD8_Tem',"CD8_Temra","CD8_Trm","CD8_Tex_1","CD8_Tex_2"])].copy()

In [ ]:
# metacell selection
metadata = pd.read_csv(f"Reproducibility/Results/SEACells/{tmp_subset}/UC_DOGMA_{tmp_subset}_metadata_w_SEACells.txt", sep='\t', index_col=0)
cell_use = np.intersect1d(adata.obs_names, metadata.index)
metadata = metadata.loc[cell_use,:]

adata_tmp = adata[cell_use,:].copy()
adata_tmp.obs['SEACell'] = metadata.loc[cell_use,'SEACell']

metacell_use = np.intersect1d(adata_tmp.obs['SEACell'], eRegulon_tmp.obs_names)
adata_tmp = adata_tmp[adata_tmp.obs['SEACell'].isin(metacell_use)].copy()

In [ ]:
# Aggregate pseudotime values
avg_pseudotime = adata_tmp.obs['dpt_pseudotime'].groupby(adata_tmp.obs['SEACell']).mean().loc[metacell_use].values
eRegulon_tmp = eRegulon_tmp[metacell_use,:]
eRegulon_tmp.obs['dpt_pseudotime'] = avg_pseudotime

In [ ]:
# Convert the original Lineage object to DataFrame
lineage_df = pd.DataFrame(
    adata_tmp.obsm['lineages_fwd'],
    index=adata_tmp.obs_names,
    columns=['CD8_Temra_2', 'CD8_Tex_2_2']
)

# Aggregate by metacells
avg_lineage_df = (
    lineage_df
    .groupby(adata_tmp.obs['SEACell'])
    .mean()
    .loc[metacell_use]  # use only the SEACells present in rna_ad
)

# Convert back to Lineage object
avg_lineages = Lineage(avg_lineage_df.values, names=avg_lineage_df.columns.tolist())

eRegulon_tmp.obsm['lineages_fwd'] = avg_lineages
eRegulon_tmp.layers['scaled'] = sc.pp.scale(eRegulon_tmp, copy=True).X

In [ ]:
model_meta = cr.models.GAMR(eRegulon_tmp, n_knots=4, smoothing_penalty=10.0)

In [ ]:
# Fig.S10G
cr.pl.gene_trends(
    eRegulon_tmp,
    model=model_meta,
    genes=['PRDM1_extended_+/+_(82g)', 'BATF_extended_+/+_(228g)', 'TBX21_extended_+/+_(128g)', 'EOMES_extended_+/+_(199g)'],
    same_plot=True,
    ncols=4,
    lineage_cmap = ["#4d9221","#c51b7d"],
    time_key="dpt_pseudotime",
    hide_cells=True,
)

plt.savefig(f"{fig_dir}FigureS10G.pdf", bbox_inches='tight')
plt.close()

In [ ]:
# Extract TFs specified by SCENIC+
import re
from collections import defaultdict

tf_list = eRegulon_tmp.var_names.tolist()
filtered = [x for x in tf_list if '_+/+_' in x]
cleaned = [re.sub(r'_[^_]+$', '', x) for x in filtered]
tf_versions = defaultdict(dict)

for item in cleaned:
    base = item.split('_')[0]
    if '_extended_+/+' in item:
        tf_versions[base]['extended'] = item
    elif '_direct_+/+' in item:
        tf_versions[base]['direct'] = item

final_tfs = []
for base, versions in tf_versions.items():
    if 'extended' in versions:
        final_tfs.append(versions['extended'])
    else:
        final_tfs.append(versions['direct'])

final_tfs = sorted(final_tfs)
print(final_tfs)

### TF activity

In [ ]:
# TF_activity
tmp_path = "Reproducibility/Results/LINGER/Primary/cell_population_TF_activity_zscore_CD8_T_NK_ILC.txt"
TF_activity = pd.read_csv(tmp_path, index_col=0, sep="\t")

cell_use = np.intersect1d(TF_activity.index, adata.obs_names)
adata_TF = sc.AnnData(TF_activity.loc[cell_use,:].copy(), obs=adata.obs.loc[cell_use,:])
adata_tmp = adata[cell_use,:].copy()

adata_TF.obsm['lineages_fwd'] = adata_tmp.obsm['lineages_fwd']

In [ ]:
model_TF = cr.models.GAMR(adata_TF, n_knots=4, smoothing_penalty=10.0)

- Exhaustion pathway

In [ ]:
target_prefixes = ['FOSB', 'SP4', 'NRF1', 'NFIA', 'MGA', 'KLF12', 'JUNB', 
            'ELF2', 'CREB1', 'LEF1', 'NFE2L3', 'RREB1', 'IKZF1', 'FOXO3',
            'MAX', 'CUX1', 'RUNX1', 'EOMES', 'TBX21', 'ELF3', 'ELF1', 'NR2C2', 
            'STAT5B', 'ZEB1', 'VDR', 'ETS1', 'ATF2', 'RUNX2', 'RUNX3', 'PPARG', 
            'BACH1', 'NFAT5', 'BATF', 'IRF2', 'MYB', 'PRDM1', 'STAT2']   # for TF activity analysis

target_prefixes2 = [prefix + '_' for prefix in target_prefixes]
selected_tfs = [gene for gene in final_tfs if gene.startswith(tuple(target_prefixes2))]
selected_tfs_final = [gene for gene in eRegulon_gene_AUC.var_names if gene.startswith(tuple(selected_tfs))]   # for eRegulon analysis

In [ ]:
# Fig.S10F
TF_order = cr.pl.heatmap(
    eRegulon_tmp,
    model=model_meta,
    data_key="X",
    genes=selected_tfs_final,
    lineages=["CD8_Tex_2_2"],
    time_key="dpt_pseudotime",
    cbar=True,
    show_all_genes=True,
    cmap = blue_yellow,
    return_genes = True
)
plt.savefig(f"{fig_dir}FigureS10F_eRegulon_exhaustion_pathway.pdf", bbox_inches='tight')
plt.close()


In [ ]:
tf_names_order = [x.split('_')[0] for x in TF_order['CD8_Tex_2_2']]

cr.pl.heatmap(
    adata_TF,
    model=model_TF,
    data_key="X",
    genes=tf_names_order,
    gene_order = tf_names_order,
    lineages=["CD8_Tex_2_2"],
    time_key="dpt_pseudotime",
    cbar=True,
    show_all_genes=True,
    cmap = solar_extra
)
plt.savefig(f"{fig_dir}FigureS10F_TF_activity_exhaustion_pathway.pdf", bbox_inches='tight')
plt.close()

- Temra pathway

In [ ]:
target_prefixes = ['FOSB', 'JUN', 'KLF12', 'NFIA', 'NFKB1', 'ELF1', 'NFE2L3', 
        'PPARG', 'LEF1', 'RORC', 'FOXO3', 'FLI1', 'CUX1', 'BACH2', 'NRF1', 
        'JUNB', 'MGA', 'POU2F1', 'SP4', 'ZEB1', 'ZNF143', 'CREB1', 
        'STAT1', 'CTCF', 'IRF3', 'ETS1', 'MYB', 
        'STAT2', 'BATF', 'ATF4', 
        'STAT5B', 'TBX21', 'PRDM1', 'NFATC2', 'NFATC3', 'NFE2L2', 
        'KLF3', 'VDR', 'BACH1','ZBTB7A', 'EOMES']

target_prefixes2 = [prefix + '_' for prefix in target_prefixes]
selected_tfs = [gene for gene in final_tfs if gene.startswith(tuple(target_prefixes2))]
selected_tfs_final = [gene for gene in eRegulon_gene_AUC.var_names if gene.startswith(tuple(selected_tfs))]

In [ ]:
# Fig.S10F
TF_order = cr.pl.heatmap(
    eRegulon_tmp,
    model=model_meta,
    data_key="X",
    genes=selected_tfs_final,
    lineages=["CD8_Temra_2"],
    time_key="dpt_pseudotime",
    cbar=True,
    show_all_genes=True,
    cmap = blue_yellow,
    return_genes = True
)
plt.savefig(f"{fig_dir}FigureS10F_eRegulon_Temra_pathway.pdf", bbox_inches='tight')
plt.close()


In [ ]:
tf_names_order = [x.split('_')[0] for x in TF_order['CD8_Temra_2']]

cr.pl.heatmap(
    adata_TF,
    model=model_TF,
    data_key="X",
    genes=tf_names_order,
    gene_order = tf_names_order,
    lineages=["CD8_Temra_2"],
    time_key="dpt_pseudotime",
    cbar=True,
    show_all_genes=True,
    cmap = solar_extra
)
plt.savefig(f"{fig_dir}FigureS10F_TF_activity_Temra_pathway.pdf", bbox_inches='tight')
plt.close()

## NK

### Diffusion pseudotime

In [ ]:
adata = adata_NK.copy()

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
palantir.preprocess.log_transform(adata)

dm_input = pd.DataFrame(adata.obsm['MultiVI_latent'], index=adata.obs_names)
dm_res = palantir.utils.run_diffusion_maps(dm_input, n_components=5)
imputed_X = palantir.utils.run_magic_imputation(adata, dm_res=dm_res)

In [ ]:
# Run diffusion maps
sc.tl.diffmap(adata)

adata_tmp = adata[adata.obs["celltype"].isin(["NK_CD56_CD49a_Lo"])].copy()
root_ixs = adata_tmp.obsm["X_diffmap"][:, 2].argmax()
root_cell = adata_tmp.obs_names[root_ixs]

adata.uns['iroot'] = np.flatnonzero(adata.obs_names == root_cell)[0]
sc.tl.dpt(adata)

In [ ]:
# Build CellRank kernels
# Pseudotime-based directional kernel
pt_kernel = PseudotimeKernel(adata, time_key='dpt_pseudotime').compute_transition_matrix()

# Similarity-based kernel from MultiVI latent
conn_kernel = ConnectivityKernel(adata).compute_transition_matrix()

# Combine kernels
tmp_ratio = 0.2
combined_kernel = tmp_ratio * conn_kernel + (1-tmp_ratio) * pt_kernel

combined_kernel.write_to_adata()

In [ ]:
sc.pl.scatter(
    adata,
    basis="diffmap",
    color=["celltype"],
    components=[1, 2],
    palette=['#3fb68b','#bc4b51','#7199c6','#e84d8a'],
    frameon=True,
    show=False
)
plt.savefig(f"{fig_dir}FigureS11C_diffmap.pdf", bbox_inches='tight')
plt.close()

In [ ]:
sc.pl.embedding(
    adata,
    basis="diffmap",
    color=['dpt_pseudotime'],
    components=["2, 3"],
    vmax=0.8,
    color_map="gnuplot",
    show = False
)
plt.savefig(f"{fig_dir}FigureS11C_pseudotime.pdf", bbox_inches='tight')
plt.close()

In [ ]:
g = cr.estimators.GPCCA(combined_kernel)
g.compute_schur()
g.compute_macrostates(n_states=4, cluster_key="celltype")
g.plot_macrostates(which="all", legend_loc="right", s=100)

In [ ]:
# Semi-automatically terminal setting
g.set_initial_states(states=["NK_CD56_CD49a_Lo"]) 
g.set_terminal_states(states=["NK_CD56_CD49a_Hi_CD103_Hi",'NK_CD56_dim'])

Compute fate probabilities

In [ ]:
g.compute_fate_probabilities()
g.plot_fate_probabilities(same_plot = True)

### eRegulon

In [ ]:
scplus_mdata = mudata.read(f"Reproducibility/Results/scenicplus/{tmp_subset}/scplusmdata.h5mu")
eRegulon_gene_AUC = ad.concat(
    [scplus_mdata["direct_gene_based_AUC"], scplus_mdata["extended_gene_based_AUC"]],
    axis = 1,
)
eRegulon_gene_AUC.obs = scplus_mdata.obs.loc[eRegulon_gene_AUC.obs_names]

In [ ]:
eRegulon_gene_AUC.obs_names = eRegulon_gene_AUC.obs_names.str.replace('___cisTopic', '', regex=False)
eRegulon_tmp = eRegulon_gene_AUC[eRegulon_gene_AUC.obs['scRNA_counts:celltype'].isin(["NK_CD56_CD49a_Hi_CD103_Hi",'NK_CD56_dim',
                                   "NK_CD56_CD49a_Hi_CD103_Lo","NK_CD56_CD49a_Lo"])].copy()

In [ ]:
# metacell selection
metadata = pd.read_csv(f"Reproducibility/Results/SEACells/{tmp_subset}/UC_DOGMA_{tmp_subset}_metadata_w_SEACells.txt", sep='\t', index_col=0)
cell_use = np.intersect1d(adata.obs_names, metadata.index)
metadata = metadata.loc[cell_use,:]

adata_tmp = adata[cell_use,:].copy()
adata_tmp.obs['SEACell'] = metadata.loc[cell_use,'SEACell']

metacell_use = np.intersect1d(adata_tmp.obs['SEACell'], eRegulon_tmp.obs_names)
adata_tmp = adata_tmp[adata_tmp.obs['SEACell'].isin(metacell_use)].copy()

In [ ]:
# Aggregate pseudotime values
avg_pseudotime = adata_tmp.obs['dpt_pseudotime'].groupby(adata_tmp.obs['SEACell']).mean().loc[metacell_use].values
eRegulon_tmp = eRegulon_tmp[metacell_use,:]
eRegulon_tmp.obs['dpt_pseudotime'] = avg_pseudotime

In [ ]:
# Convert the original Lineage object to DataFrame
lineage_df = pd.DataFrame(
    adata_tmp.obsm['lineages_fwd'],
    index=adata_tmp.obs_names,
    columns=["NK_CD56_CD49a_Hi_CD103_Hi",'NK_CD56_dim']
)

# Aggregate by metacells
avg_lineage_df = (
    lineage_df
    .groupby(adata_tmp.obs['SEACell'])
    .mean()
    .loc[metacell_use]  # use only the SEACells present in rna_ad
)

# Convert back to Lineage object
avg_lineages = Lineage(avg_lineage_df.values, names=avg_lineage_df.columns.tolist())

eRegulon_tmp.obsm['lineages_fwd'] = avg_lineages
eRegulon_tmp.layers['scaled'] = sc.pp.scale(eRegulon_tmp, copy=True).X

In [ ]:
model_meta = cr.models.GAMR(eRegulon_tmp, n_knots=3, smoothing_penalty=10.0)

In [ ]:
# Fig.S11F
cr.pl.gene_trends(
    eRegulon_tmp,
    model=model_meta,
    genes=['PRDM1_extended_+/+_(82g)', 'RUNX1_extended_+/+_(668g)', 'TBX21_extended_+/+_(128g)'],
    same_plot=True,
    ncols=3,
    lineage_cmap = ["#d73027","#4575b4"],
    time_key="dpt_pseudotime",
    hide_cells=True,
)

plt.savefig(f"{fig_dir}FigureS11F.pdf", bbox_inches='tight')
plt.close()

In [ ]:
# Extract TFs specified by SCENIC+
import re
from collections import defaultdict

tf_list = eRegulon_tmp.var_names.tolist()
filtered = [x for x in tf_list if '_+/+_' in x]
cleaned = [re.sub(r'_[^_]+$', '', x) for x in filtered]
tf_versions = defaultdict(dict)

for item in cleaned:
    base = item.split('_')[0]
    if '_extended_+/+' in item:
        tf_versions[base]['extended'] = item
    elif '_direct_+/+' in item:
        tf_versions[base]['direct'] = item

final_tfs = []
for base, versions in tf_versions.items():
    if 'extended' in versions:
        final_tfs.append(versions['extended'])
    else:
        final_tfs.append(versions['direct'])

final_tfs = sorted(final_tfs)
print(final_tfs)

### TF activity

In [ ]:
# TF_activity
tmp_path = "Reproducibility/Results/LINGER/Primary/cell_population_TF_activity_zscore_CD8_T_NK_ILC.txt"
TF_activity = pd.read_csv(tmp_path, index_col=0, sep="\t")

cell_use = np.intersect1d(TF_activity.index, adata.obs_names)
adata_TF = sc.AnnData(TF_activity.loc[cell_use,:].copy(), obs=adata.obs.loc[cell_use,:])
adata_tmp = adata[cell_use,:].copy()

adata_TF.obsm['lineages_fwd'] = adata_tmp.obsm['lineages_fwd']

In [ ]:
model_TF = cr.models.GAMR(adata_TF, n_knots=3, smoothing_penalty=10.0)

- Adaptive-like pathway

In [ ]:
target_prefixes = ['FOSB', 'ELF2', 'EOMES', 'IRF8', 'JUNB', 'KLF12', 'KLF3', 'RREB1', 
 'EGR1', 'NFE2L3', 'NFIA', 'NFKB1', 'REL', 'LEF1', 'RORA', 'FOXO3', 'SP4', 'CUX1', 'TBX21', 'NFE2L2', 'CREB1', 
 'IKZF1', 'JUND', 'STAT5B', 'RUNX1', 'JUN', 'STAT1', 'BATF', 'NFATC3', 'IRF3',
 'YY1', 'ELF1', 'FOXO1', 'RUNX3', 'RUNX2', 'BACH1', 'IRF2', 'RFX2', 'TCFL5', 
 'PRDM1', 'PPARG', 'MYB', 'POU2F1', 'ZEB1', 'STAT2', 'ETS1', 'BHLHE40']   # for TF activity analysis

target_prefixes2 = [prefix + '_' for prefix in target_prefixes]
selected_tfs = [gene for gene in final_tfs if gene.startswith(tuple(target_prefixes2))]
selected_tfs_final = [gene for gene in eRegulon_gene_AUC.var_names if gene.startswith(tuple(selected_tfs))]   # for eRegulon analysis

In [ ]:
# Fig.S11D
TF_order = cr.pl.heatmap(
    eRegulon_tmp,
    model=model_meta,
    data_key="X",
    genes=selected_tfs_final,
    lineages=["NK_CD56_CD49a_Hi_CD103_Hi"],
    time_key="dpt_pseudotime",
    cbar=True,
    show_all_genes=True,
    cmap = blue_yellow,
    return_genes = True
)
plt.savefig(f"{fig_dir}FigureS11D_eRegulon_adaptive-like_pathway.pdf", bbox_inches='tight')
plt.close()

In [ ]:
tf_names_order = [x.split('_')[0] for x in TF_order["NK_CD56_CD49a_Hi_CD103_Hi"]]

cr.pl.heatmap(
    adata_TF,
    model=model_TF,
    data_key="X",
    genes=tf_names_order,
    gene_order = tf_names_order,
    lineages=["NK_CD56_CD49a_Hi_CD103_Hi"],
    time_key="dpt_pseudotime",
    cbar=True,
    show_all_genes=True,
    cmap = solar_extra
)
plt.savefig(f"{fig_dir}FigureS11D_TF_activity_adaptive-like_pathway.pdf", bbox_inches='tight')
plt.close()

- Maturation pathway

In [ ]:
target_prefixes = ['FOSB', 'REL', 'ELF1', 'NR2C2', 'PPARG', 'NFKB1', 'IRF8', 'FOXO3', 'CUX1', 
 'BACH2', 'LEF1', 'RORC', 'POU2F1', 'ATF2', 'ZEB1', 'FOXO1', 'RUNX2', 
 'REST', 'TCFL5', 'BACH1', 'STAT2', 'STAT1', 'PRDM1', 'JUND', 'JUN', 
 'TBX21', 'STAT5B', 'RUNX1', 'MGA', 'KLF3', 'IKZF1', 'IRF2', 'MAX', 'NFATC2', 'NFATC3', 
 'CREB1', 'KLF5', 'EGR1', 'ELF3', 'EOMES', 'FLI1', 'ZBTB7A', 'IRF3', 'JUNB']  # for TF activity analysis

target_prefixes2 = [prefix + '_' for prefix in target_prefixes]
selected_tfs = [gene for gene in final_tfs if gene.startswith(tuple(target_prefixes2))]
selected_tfs_final = [gene for gene in eRegulon_gene_AUC.var_names if gene.startswith(tuple(selected_tfs))]   # for eRegulon analysis

In [ ]:
# Fig.S11D
TF_order = cr.pl.heatmap(
    eRegulon_tmp,
    model=model_meta,
    data_key="X",
    genes=selected_tfs_final,
    lineages=["NK_CD56_dim"],
    time_key="dpt_pseudotime",
    cbar=True,
    show_all_genes=True,
    cmap = blue_yellow,
    return_genes = True
)
plt.savefig(f"{fig_dir}FigureS11D_eRegulon_maturation_pathway.pdf", bbox_inches='tight')
plt.close()

In [ ]:
tf_names_order = [x.split('_')[0] for x in TF_order["NK_CD56_dim"]]

cr.pl.heatmap(
    adata_TF,
    model=model_TF,
    data_key="X",
    genes=tf_names_order,
    gene_order = tf_names_order,
    lineages=["NK_CD56_dim"],
    time_key="dpt_pseudotime",
    cbar=True,
    show_all_genes=True,
    cmap = solar_extra
)
plt.savefig(f"{fig_dir}FigureS11D_TF_activity_maturation_pathway.pdf", bbox_inches='tight')
plt.close()